# 01 — Data Quality Review

Lightweight walkthrough of validation results for **Brew & Bloom Coffee Co.** synthetic data.

**Core logic lives in** `src/validate_data.py` — this notebook reads precomputed outputs after `make validate`.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from config import STUDY_END, STUDY_START

TABLES = ROOT / "outputs" / "tables"
PROCESSED = ROOT / "data" / "processed"

## Validation summary

The pipeline runs **33 automated checks** and writes `validation_summary.csv` plus `validation_report.json`.

In [ ]:
summary = pd.read_csv(TABLES / "validation_summary.csv")
report = json.loads((PROCESSED / "validation_report.json").read_text())

print(f"Status: {report['status'].upper()}")
print(f"Fails: {report['fail_checks']} | Warnings: {report['warn_checks']}")
print(f"Study period: {STUDY_START} → {STUDY_END}")
print("\nRecord counts:")
for name, count in report["record_counts"].items():
    print(f"  {name}: {count:,}")

print("\nQuality metrics:")
for k, v in report["quality_metrics"].items():
    print(f"  {k}: {v}")

summary["status"].value_counts()

In [ ]:
summary.sort_values(["status", "check_name"])

## Duplicate checks

Primary keys must be unique across stores, surveys, comments, product feedback, and loyalty guests.

In [ ]:
dup_checks = summary[summary["check_name"].str.startswith("unique_")]
dup_checks[["check_name", "status", "issue_count", "notes"]]

## Range checks

NPS must be 0–10; CSAT, revisit intent, and experience ratings must be 1–5.

In [ ]:
range_checks = summary[summary["check_name"].str.contains("range")]
range_checks[["check_name", "status", "records_checked", "issue_count", "notes"]]

## Missing values & referential integrity

Required fields populated; foreign keys link surveys → stores, comments → surveys, loyalty → guests.

In [ ]:
missing_checks = summary[
    summary["check_name"].str.contains("required_fields|exists|without_comments", regex=True)
]
missing_checks[["check_name", "status", "issue_count", "notes"]]

## Suspicious record examples

Warnings flag straight-lining, inconsistent promoter scores, and surveys without linked comments. Examples are exported to `flagged_records.csv` (logic in `src/validate_data.py`).

In [ ]:
flagged = pd.read_csv(PROCESSED / "flagged_records.csv")
warn_checks = summary[summary["status"] == "warn"]
warn_checks[["check_name", "issue_count", "notes"]]

In [ ]:
flagged["flag_reason"].value_counts()

In [ ]:
surveys = pd.read_parquet(PROCESSED / "guest_surveys_clean.parquet")
sample_ids = flagged["survey_id"].dropna().head(3).tolist()
experience_cols = [
    "wait_time_rating", "drink_quality_rating", "order_accuracy_rating",
    "staff_friendliness_rating", "cleanliness_rating",
    "mobile_app_experience_rating", "rewards_satisfaction", "price_value_perception",
]
surveys.loc[surveys["survey_id"].isin(sample_ids), ["survey_id", "nps", "csat", "revisit_intent"] + experience_cols]

## Takeaway

- All **fail-level** checks pass — data is analysis-ready.
- **58 flagged records** are retained with warning flags for transparency.
- Re-run validation anytime: `make validate` or `.venv/bin/python src/validate_data.py`.